# 蒙特卡洛基础方法（MC Basic）

> ## 为什么蒙特卡洛是"无模型"方法？
>
> **有模型方法**（如策略迭代、值迭代）需要预先知道环境的**数学模型**——即**状态转移概率** $P(s'|s,a)$ 和**奖励函数** $R(s,a,s')$ 的解析表达式，才能用贝尔曼方程直接计算期望：
>
> $$V(s) = \sum_{a} \pi(a|s) \sum_{s'} P(s'|s,a)\left[R(s,a,s') + \gamma V(s')\right]$$
>
> **蒙特卡洛方法**既不需要 $P(s'|s,a)$ 的公式，也不需要 $R(s,a,s')$ 的公式。它直接让智能体与环境交互，跑完整的 Episode，通过**实际采样**观测到奖励值，再用累积回报估计状态价值：
>
> $$G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \cdots$$
>
> $$V(s) \approx \frac{1}{N}\sum_{i=1}^{N} G_t^{(i)} \bigg|_{S_t=s}$$
>
> 由**大数定理**保证：采样次数足够多时，样本均值收敛到真实期望。
>
> ⚠️ **注意**：MC 确实会收到即时奖励 $R$，但那是通过**实际执行动作后环境反馈的观测值**，而非预先掌握的数学公式。
>
> ---

| | 有模型 | 蒙特卡洛（无模型） |
|--|--|--|
| $P(s' \mid s,a)$ | ✅ 需要预先知道解析公式 | ❌ 不需要，靠采样观测转移结果 |
| $R(s,a,s')$ | ✅ 需要预先知道解析公式 | ❌ 不需要公式，靠实际交互获得奖励数值 |
| 是否需要跑 Episode | ❌ 不需要，直接查公式计算 | ✅ 必须跑完整 Episode |

> ---
>
> **直觉类比：**
>
> ---

| 方法 | 类比 |
|------|------|
| 有模型 | 拆开老虎机，**预先知道**每个结果的概率和奖励，不用投币就能算期望 |
| 蒙特卡洛 | 不知道机器内部构造，**真正投币**，记录每次奖励，统计平均值 |

> ---
>
> "无模型"的本质是：**无需事先建立环境的数学模型**，把环境当成黑盒，只通过与环境的真实交互来学习。

## 一、导入依赖库

In [5]:
import numpy as np       # 导入 NumPy 库，用于数值计算和矩阵运算，版本要求 >=1.18
import random            # 导入 Python 标准库 random，用于随机数生成
import importlib.util    # 导入 importlib.util，用于按文件路径动态加载模块

# 文件名 "02.1.ModelFree_Env_GridWorldV2.py" 以数字开头且含点号，不符合 Python 标识符规则，无法直接 import
# spec_from_file_location：根据给定模块别名和 .py 文件路径创建模块规格，返回 ModuleSpec 对象
_spec = importlib.util.spec_from_file_location("GridWorld_v2", "02.1.ModelFree_Env_GridWorldV2.py")
# module_from_spec：根据模块规格创建模块对象，此时模块代码尚未执行，返回 module 对象
GridWorld_v2 = importlib.util.module_from_spec(_spec)
# exec_module：执行模块代码完成初始化，之后可通过 GridWorld_v2.GridWorld_v2(...) 正常使用类
_spec.loader.exec_module(GridWorld_v2)

In [6]:
np.eye(5)  # 生成 5×5 单位矩阵，np.ndarray，shape=(5,5)，对角线为 1 其余为 0；演示 one-hot 编码的矩阵基础

array([[1., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0.],
       [0., 0., 1., 0., 0.],
       [0., 0., 0., 1., 0.],
       [0., 0., 0., 0., 1.]])

## 二、初始化网格世界与策略

In [7]:
gamma = 0.9   # 折扣因子 γ，float，控制未来奖励的衰减程度，越接近 0 越短视，越接近 1 越重视长期回报

rows = 5      # 网格世界的行数，int，需与 desc 描述字符串的行数保持一致
columns = 5   # 网格世界的列数，int，需与 desc 描述字符串每行字符数保持一致

# 使用描述字符串初始化网格世界：'.' 表示普通格，'#' 表示禁止区（得分 -10），'T' 表示目标格（得分 1）
gridworld = GridWorld_v2.GridWorld_v2(
    forbiddenAreaScore=-10,  # 禁止区域的即时奖励，float，负值表示惩罚
    score=1,                 # 目标区域的即时奖励，float
    desc=[".....", ".##..", "..#..", ".#T#.", ".#..."]  # 网格布局描述，list[str]，共 5 行 5 列
)
gridworld.show()  # 以 emoji 可视化打印网格世界布局，无返回值

trajectorySteps = 100  # 每次采样轨迹的最大步数，int

value = np.zeros(rows * columns)        # 初始化状态价值函数 V(s)，np.ndarray，shape=(25,)，全零
qtable = np.zeros((rows * columns, 5))  # 初始化动作价值函数 Q(s,a)，np.ndarray，shape=(25, 5)，全零

# 随机初始化确定性策略，利用 NumPy 花式索引（Fancy Indexing）一步完成整数索引→one-hot 转换：
# np.random.randint(0,5,size=(rows*columns))：生成 25 个随机动作索引，shape=(25,)，每个值∈[0,4]，int
# np.eye(5)[整数数组]：花式索引——对数组中每个整数 i，取 np.eye(5) 第 i 行（动作 i 的 one-hot 向量）
#   并沿第 0 轴堆叠，等价于 np.stack([np.eye(5)[i] for i in idx])，但由 C 实现、效率更高
# 结果 policy：np.ndarray，shape=(25, 5)，每行恰有一个 1（对应该状态随机选定的动作），其余为 0
policy = np.eye(5)[np.random.randint(0, 5, size=(rows * columns))]
print(policy)  # 打印初始随机策略矩阵，shape=(25, 5)，每行恰有一个 1，其余为 0

gridworld.showPolicy(policy)  # 以 emoji 可视化当前策略，显示每个格子的推荐动作

⬜️⬜️⬜️⬜️⬜️
⬜️🚫🚫⬜️⬜️
⬜️⬜️🚫⬜️⬜️
⬜️🚫✅🚫⬜️
⬜️🚫⬜️⬜️⬜️
[[1. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 0. 1. 0.]
 [1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 1.]
 [0. 0. 1. 0. 0.]
 [1. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0.]
 [0. 0. 1. 0. 0.]
 [0. 1. 0. 0. 0.]
 [1. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 1.]
 [1. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 1.]
 [0. 1. 0. 0. 0.]
 [1. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 1.]]
⬆️⬆️⬆️⬇️⬅️
⬆️🔄🔄⬇️⬆️
⬅️⬇️⏩️⬆️⬇️
➡️🔄✅⏪🔄
➡️⏫️➡️🔄🔄


## 三、MC Basic 策略迭代

In [8]:
gridworld.showPolicy(policy)  # 显示当前（随机初始化）策略在网格世界中的动作分布
print("初始随机策略（迭代前）")  # 打印提示：当前为随机策略，即迭代前的初始状态

qtable = np.zeros((rows * columns, 5))  # 重新初始化 Q 表，np.ndarray，shape=(25, 5)，全零
qtable_pre = qtable.copy() + 1          # 复制 Q 表并整体加 1 作为上一轮快照，保证首轮进入循环，shape=(25, 5)

iteration = 0  # 迭代计数器，int，记录当前策略迭代轮次，初始为 0

# 收敛判断：当前轮与上一轮 Q 表元素差的平方和大于 0.001 时继续迭代
while np.sum((qtable_pre - qtable) ** 2) > 0.001:
    iteration += 1  # 每进入一轮迭代，计数器加 1
    print(f"\n{'=' * 50}")  # 打印分隔线，标识新一轮迭代的开始
    print(f"  第 {iteration} 轮迭代  |  迭代开始 Q 差异：{np.sum((qtable_pre - qtable) ** 2):.6f}")  # 打印轮次和当前 Q 表变化量
    print(f"{'=' * 50}")  # 打印分隔线下边框

    qtable_pre = qtable.copy()  # 将当前 Q 表保存为上一轮快照，np.ndarray，shape=(25, 5)

    for i in range(rows * columns):  # 遍历所有状态，i 为状态编号，int，范围 [0, 25)
        for j in range(5):           # 遍历所有动作，j 为动作编号，int，范围 [0, 5)
            # 从状态 i 出发执行动作 j，按当前策略 policy 采样 trajectorySteps 步轨迹
            # 返回值 Trajectory：list[tuple]，长度为 trajectorySteps+1
            # 每个元组格式：(nowState, nowAction, score, nextState, nextAction)
            Trajectory = gridworld.getTrajectoryScore(
                nowState=i, action=j, policy=policy, steps=trajectorySteps
            )

            # 取轨迹末尾（第 trajectorySteps 步）的即时奖励作为折扣回报递推的初始值，float
            tmp = Trajectory[trajectorySteps][2]

            # 【逆向递推计算折扣累计回报】
            # 折扣累计回报展开式：G_0 = r_0 + γ·r_1 + γ²·r_2 + ... + γ^T·r_T
            # 若正向遍历，每步须单独计算 γ^k（幂运算），总计 O(T²) 次乘法，效率低
            # 利用递推关系 G_k = r_k + γ·G_{k+1}，从末尾 G_T 出发逆向推算：
            #   G_T = r_T
            #   G_{T-1} = r_{T-1} + γ·G_T
            #   G_{T-2} = r_{T-2} + γ·G_{T-1}  ...  G_0 = r_0 + γ·G_1
            # 每步仅需一次乘法和一次加法，总计 O(T) 次乘法，远优于正向遍历的 O(T²)
            for k in range(trajectorySteps - 1, -1, -1):
                # Trajectory[k][2]：第 k 步即时奖励 r_k，float
                # 迭代前 tmp = G_{k+1}，迭代后 tmp = G_k = r_k + γ·G_{k+1}，float
                tmp = tmp * gamma + Trajectory[k][2]  # G_k = r_k + γ·G_{k+1}

            # 将状态-动作对 (i, j) 的折扣累计回报写入 Q 表：Q(s=i, a=j) ← G
            qtable[i][j] = tmp

    # 【贪心策略改进】对每个状态选取 Q 值最大的动作，构造新的确定性策略（one-hot 编码）
    # 分两步理解：
    #   第一步 np.argmax(qtable, axis=1)：
    #     qtable shape=(25,5)，axis=1 表示在每行（每个状态）内找最大 Q 值的列索引
    #     返回 shape=(25,) 的整数数组，每个元素∈[0,4]，即该状态的最优动作编号
    #     例：状态0的Q值为[-1, -1, 9, -1, 10] → argmax=4，即动作4最优
    #   第二步 np.eye(5)[整数数组]：
    #     np.eye(5) 是 5×5 单位矩阵，第 i 行 = 动作 i 的 one-hot 向量
    #       第0行→[1,0,0,0,0]，第1行→[0,1,0,0,0]，...，第4行→[0,0,0,0,1]
    #     用整数数组批量取行：动作编号是几就取第几行，等价于逐状态构造 one-hot
    #     结果 policy：np.ndarray，shape=(25, 5)，每行恰有一个 1（最优动作位置），其余为 0
    policy = np.eye(5)[np.argmax(qtable, axis=1)]

    q_diff = np.sum((qtable_pre - qtable) ** 2)  # 计算本轮 Q 表变化量（平方和），float
    print(f"  状态17 Q值：{qtable[17]}")  # 打印状态 17 的 Q 值向量，np.ndarray，shape=(5,)，用于调试
    print(f"  状态22 Q值：{qtable[22]}")  # 打印状态 22 的 Q 值向量，np.ndarray，shape=(5,)，用于调试
    print(f"  更新后策略：")  # 打印策略可视化标签
    gridworld.showPolicy(policy)  # 可视化当前更新后的策略
    print(f"  本轮 Q 差异：{q_diff:.6f}  {'（已收敛）' if q_diff <= 0.001 else '（继续迭代）'}")  # 打印本轮变化量及收敛状态

print(f"\n{'*' * 50}")  # 打印最终分隔线
print(f"  策略迭代完成，共迭代 {iteration} 轮")  # 打印迭代总轮次
print(f"{'*' * 50}")


⬆️⬆️⬆️⬇️⬅️
⬆️🔄🔄⬇️⬆️
⬅️⬇️⏩️⬆️⬇️
➡️🔄✅⏪🔄
➡️⏫️➡️🔄🔄
初始随机策略（迭代前）

  第 1 轮迭代  |  迭代开始 Q 差异：125.000000
  状态17 Q值：[-10.         -17.2          0.         -99.99760947  -8.        ]
  状态22 Q值：[ -8.           0.          -1.         -99.99760947   0.        ]
  更新后策略：
⬇️➡️➡️➡️⬇️
🔄⏪⏩️⬆️⬆️
⬆️⬅️⏩️⬆️⬆️
⬆️⏩️✅⏫️⬆️
⬆️⏩️➡️➡️⬆️
  本轮 Q 差异：324677.354073  （继续迭代）

  第 2 轮迭代  |  迭代开始 Q 差异：324677.354073
  状态17 Q值：[-10.  -10.    0.   -9.1   1. ]
  状态22 Q值：[  1.   0.  -1. -10.   0.]
  更新后策略：
➡️➡️➡️➡️⬇️
⬆️⏫️⏫️⬆️⬆️
⬆️⬅️⏬⬆️⬆️
⬆️⏩️✅⏪⬆️
⬆️⏩️⬆️➡️⬆️
  本轮 Q 差异：286911.112951  （继续迭代）

  第 3 轮迭代  |  迭代开始 Q 差异：286911.112951
  状态17 Q值：[-1.00023905 -1.00023905  8.99976095 -1.00023905  9.99976095]
  状态22 Q值：[ 9.99976095  0.          7.99976095 -1.90023905  8.99976095]
  更新后策略：
➡️➡️➡️➡️⬇️
⬆️⏫️⏫️⬆️⬆️
⬆️⬅️⏬⬆️⬆️
⬆️⏩️✅⏪⬆️
⬆️⏩️⬆️⬅️⬆️
  本轮 Q 差异：2275.975216  （继续迭代）

  第 4 轮迭代  |  迭代开始 Q 差异：2275.975216
  状态17 Q值：[-1.00023905 -1.00023905  8.99976095 -1.00023905  9.99976095]
  状态22 Q值：[ 9.99976095  8.09976095  7.99976095 -1.90023905  8.99